In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import pyro
import pyro.distributions as dist

from pyro.infer import MCMC
from pyro.infer.mcmc import RandomWalkKernel

from autoemulate.simulations.epidemic import Epidemic
from autoemulate.simulations.seir import SEIRSimulator
from autoemulate.core.compare import AutoEmulate
from autoemulate.emulators import GaussianProcessRBF
from autoemulate.calibration.bayes import BayesianCalibration
from autoemulate.calibration.evidence import EvidenceComputation

seed = 123
torch.manual_seed(seed)
np.random.seed(seed)
pyro.set_rng_seed(seed)
true_beta  = 0.3
true_gamma = 0.15
true_sigma = 0.2

noise_std = 0.05
n_obs = 100

seir_sim = SEIRSimulator(log_level="error")

params = torch.tensor([true_beta, true_gamma, true_sigma]).view(1, -1)
true_output = seir_sim.forward(params)

noise = torch.normal(0, noise_std, size=(n_obs,))
observed = true_output[0] + noise

observations = {"infection_rate": observed}
n_train = 1500

# SEIR training data
x_seir = seir_sim.sample_inputs(n_train)
y_seir, _ = seir_sim.forward_batch(x_seir)

ae_seir = AutoEmulate(
    x_seir,
    y_seir,
    models=[GaussianProcessRBF],
    device="cpu",
    log_level="error"
)

gp_seir = ae_seir.best_result().model

sir_sim = Epidemic(log_level="error")

x_sir = sir_sim.sample_inputs(n_train)
y_sir, _ = sir_sim.forward_batch(x_sir)

ae_sir = AutoEmulate(
    x_sir,
    y_sir,
    models=[GaussianProcessRBF],
    device="cpu",
    log_level="error"
)

gp_sir = ae_sir.best_result().model

warmup = 500
samples = 1500
chains = 2

bc_seir = BayesianCalibration(
    gp_seir,
    seir_sim.parameters_range,
    observations,
    observation_noise=noise_std**2
)

mcmc_seir = bc_seir.run_mcmc(
    warmup_steps=warmup,
    num_samples=samples,
    num_chains=chains
)

bc_sir = BayesianCalibration(
    gp_sir,
    sir_sim.parameters_range,
    observations,
    observation_noise=noise_std**2
)

mcmc_sir = bc_sir.run_mcmc(
    warmup_steps=warmup,
    num_samples=samples,
    num_chains=chains
)

training_prop = 0.5

ec_seir = EvidenceComputation(
    mcmc_seir,
    bc_seir.model,
    training_proportion=training_prop
)

res_seir = ec_seir.run(epochs=100, verbose=False)

ec_sir = EvidenceComputation(
    mcmc_sir,
    bc_sir.model,
    training_proportion=training_prop
)

res_sir = ec_sir.run(epochs=100, verbose=False)

lnZ_seir = res_seir["ln_evidence"]
lnZ_sir  = res_sir["ln_evidence"]

bayes_factor = np.exp(lnZ_seir - lnZ_sir)
print("\nSIR Model Posterior Summary:")
sir_summary = mcmc_sir.summary()
print(sir_summary)

print("\nSEIR Model Posterior Summary:")
seir_summary = mcmc_seir.summary()
print(seir_summary)
print("SEIR ln(Z):", lnZ_seir)
print("SIR  ln(Z):", lnZ_sir)
print("Bayes Factor (SEIR / SIR):", bayes_factor)